# DTMIR-Pro+ — train on Colab (GPU)

This notebook **downloads the SOTS (haze) dataset automatically**, then runs the
**B0–B4 ablation ladder** (baseline reproduction + our three changes).

It produces:
- `runs/ablation/summary.csv` — one PSNR/SSIM per ablation row
- per-row training curves (`runs/ablation/<row>_curves.png`)

Run the cells **top to bottom**. The first real run uses a subset of SOTS (about
400 image pairs) so it finishes quickly; bump the epochs later for the full result.

## Step 1 — Turn on the GPU

Menu: **Runtime → Change runtime type → Hardware accelerator = GPU**, then **Save**.
The paper used a GTX 1080 8GB, so the free Colab T4 is plenty.

In [ ]:
import shutil, subprocess, torch
if shutil.which('nvidia-smi'):
    print('GPU:', subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                                 capture_output=True, text=True).stdout.strip())
else:
    print('No GPU detected. Turn it on via Runtime -> Change runtime type -> GPU, then re-run.')
print('CUDA available:', torch.cuda.is_available())

## Step 2 — Install dependencies

In [ ]:
!pip install -q torch torchvision torchmetrics pyyaml pillow numpy

## Step 3 — Upload the code

On your computer, **zip the `dtmir_pro_plus` folder** (right-click → Send to →
Compressed folder). Then run the cell below and choose that zip.

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()   # pick dtmir_pro_plus.zip
for name in uploaded:
    with zipfile.ZipFile(name, 'r') as z:
        z.extractall('/content')
print('Uploaded and extracted:', os.listdir('/content'))

In [ ]:
import os
os.chdir('/content/dtmir_pro_plus')
!python sanity_check.py

## Step 4 — Download the SOTS dataset (automatic)

Downloads a 435 MB SOTS subset from Hugging Face, then arranges it into the folder
layout the training code expects.

In [ ]:
import os, pathlib, subprocess, sys

url = "https://huggingface.co/datasets/desonglll/reside-SOTS/resolve/main/SOTS.zip"
if not os.path.exists("SOTS.zip"):
    print("Downloading SOTS.zip (~435 MB)...")
    subprocess.run(["wget", "-q", "-O", "SOTS.zip", url], check=True)
if not os.path.exists("sots_raw"):
    subprocess.run(["unzip", "-q", "-o", "SOTS.zip", "-d", "sots_raw"], check=True)

base = pathlib.Path("sots_raw/SOTS")
split = "outdoor" if (base / "outdoor").is_dir() else \
        ("indoor" if (base / "indoor").is_dir() else None)
assert split, "Could not find the SOTS outdoor/indoor folders after unzipping"
print("Using SOTS split:", split)

res = subprocess.run([sys.executable, "-m", "data.prepare_sots",
                       "--sots-root", str(base / split), "--outdir", "data/sots",
                       "--size", "256", "--num", "400", "--seed", "0"],
                       capture_output=True, text=True)
print(res.stdout)
print(res.stderr)
res.check_returncode()
print("\nPrepared data/sots (restoration pairs + synthesized rain/snow translation targets)")

## Step 5 — Run the B0–B4 ablation

Quick settings below (finishes faster). For the paper's full schedule set
`EPOCHS_STAGE1 = 100` and `EPOCHS_STAGE2 = 300`.

In [ ]:
DATA_ROOT = 'data/sots'
EPOCHS_STAGE1 = 30
EPOCHS_STAGE2 = 80
IMAGE_SIZE = 256
BATCH_SIZE = 8

!python run_ablation.py --data-root {DATA_ROOT} --image-size {IMAGE_SIZE} \
    --batch-size {BATCH_SIZE} --epochs-stage1 {EPOCHS_STAGE1} \
    --epochs-stage2 {EPOCHS_STAGE2} --use-vgg 1 --outdir runs/ablation

## Step 6 — See the comparison table

In [ ]:
import pandas as pd
df = pd.read_csv('runs/ablation/summary.csv')
display(df)
print('Paper reference (SOTS, full dataset): PSNR 36.92, SSIM 0.981')

In [ ]:
for row in ['B0','B1','B2','B3','B4']:
    !python plot_training.py --csv runs/ablation/{row}/metrics.csv \
        --out runs/ablation/{row}_curves.png --title "DTMIR-Pro+ {row}"

## Notes / known caveats

- This is a **SOTS subset** (~400 pairs), not the full 5000-image RESIDE SOTS, so
  the numbers will land below the paper's 36.92/0.981. Treat it as a first real
  curve, then point at the full dataset.
- The translation stage's rain/snow targets are **synthesized** (real ORD/CSD are
  not bundled) — documented simplification for the first haze run.
- The contrastive loss (`losses.py`) can spike early in training; flagged and to be
  re-checked against the paper before trusting final numbers.

### Optional: quick smoke test with fake data (no download)
```python
!python data/make_synthetic_data.py --num 200 --size 128 --seed 0
!python run_ablation.py --data-root data/synthetic --image-size 128 \
    --batch-size 8 --epochs-stage1 5 --epochs-stage2 10 --use-vgg 0 --outdir runs/smoke
```